In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./data/output_from_KNIME/binding-site.csv")
print(df.shape)
df.head()

In [ ]:
'''
df= (
    df.groupby("type", group_keys=False)
      .apply(lambda x: x.sample(frac=0.1, random_state=42))
)
'''
df.shape


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_filtered = df.copy()
df_filtered = df_filtered[df_filtered["minimum_bound_fraction"].notna()]
df_filtered = df_filtered[df_filtered["binding_site"].notna()]
df_filtered["selectivity-label"] = df_filtered["selectivity-label"].astype(str)

palette = {
    "selective": "Aquamarine",
    "unselective": "LightSeaGreen"
}
site_palette = dict(zip(df_filtered["binding_site"].unique(), sns.color_palette("tab10", df_filtered["binding_site"].nunique())))

plt.figure(figsize=(10, 5))
ax = sns.stripplot(
    x="binding_site", y="minimum_bound_fraction",
    data=df_filtered, hue="selectivity-label",
    dodge=True, jitter=0.25, palette=palette
)

plt.axhline(0.4, linestyle="--", color="gray", alpha=0.5, label="threshold=0.4")

plt.title("Binding Strength by Binding Site and Selectivity")
plt.ylabel("Binding Score")
plt.xlabel("Binding Site")
plt.grid(False)

plt.legend(
    title="Selectivity",
    bbox_to_anchor=(1.02, 0),  # 右下（x > 1.0, y = 0.0）
    loc="lower left",
    borderaxespad=0,
    frameon=False
)

plt.tight_layout()
plt.show()


In [ ]:
# 強力かつ選択的なものだけ抽出
strong_selective = df[
    (df["minimum_bound_fraction"] > 0.4) &
    (df["selectivity-label"] == "selective")
]

# その他のデータ
others = df[~((df["minimum_bound_fraction"] > 0.4) &
              (df["selectivity-label"] == "selective"))]


In [ ]:
import pandas as pd
from scipy.stats import fisher_exact

# クロス集計表（2×2）
table = pd.DataFrame({
    "multiloop": [
        (strong_selective["binding_site"] == "multiloop").sum(),
        (others["binding_site"] == "multiloop").sum()
    ],
    "not_multiloop": [
        (strong_selective["binding_site"] != "multiloop").sum(),
        (others["binding_site"] != "multiloop").sum()
    ]
}, index=["strong_selective", "others"])

# Fisherの正確確率検定
oddsratio, p_value = fisher_exact(table)
print(table)
print(f"p-value = {p_value:.4f}")